In [3]:
import pandas as pd
import numpy as np
import re

#ploting
import matplotlib.pyplot as plt
import seaborn as sns

#train_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Bidirectional,LSTM,Dense,Dropout



#Reading csv file


In [6]:
import pandas as pd

# Load spam dataset
spam = pd.read_csv("spam.csv", encoding="latin1")
spam = spam[['v1','v2']]              # pick only relevant columns
spam.columns = ['label','text']       # rename

# Load feedback dataset
feedback = pd.read_csv("feedback.csv", encoding="latin1")
feedback = feedback[['label','message']]  # pick relevant columns
feedback.columns = ['label','text']       # rename to match spam

# Concatenate
df = pd.concat([spam, feedback], ignore_index=True)

# Ensure label column is string
df['label'] = df['label'].astype(str)

# Lowercase + map to 0/1
df['label'] = df['label'].str.lower().map({'ham':0,'spam':1})

# Drop rows with NaN label
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(df.head())

   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [8]:
df.shape

(5594, 2)

#finding missing values


In [5]:
# # Rename both columns correctly
# df = df.rename(columns={'v1':'label', 'v2':'text'})

# # Keep only label + text
# df = df[['label', 'text']]

# # Ensure all labels are strings first
# df['label'] = df['label'].astype(str)

# # Convert to lowercase and map to 0/1
# df['label'] = df['label'].str.lower().map({'ham':0,'spam':1})

# # Drop rows where label becamae NaN (invalid/missing)
# df = df.dropna(subset=['label'])

# # Convert to integer type
# df['label'] = df['label'].astype(int)

# print(df.head())
# print(df['label'].unique())  # should show only [0,1]


Empty DataFrame
Columns: [label, text]
Index: []
[]


In [9]:
#clean and preprocess text (removing emoji and converting unnecessary things and to lowercase)
def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+","",text)
    text=re.sub(r"[^a-zA-Z0-9\s]","",text)
    return text


df['cleaned_text']=df['text'].apply(clean_text)

df.head()


,label,text,cleaned_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [10]:
tokenizer=Tokenizer(num_words=5000,oov_token='<OOV>')
#Creates a Tokenizer object that converts words to numbers.
#It will keep only the top 10,000 most frequent words.
#Any unknown (rare) word will be replaced with the token <OOV> (out-of-vocabulary).

tokenizer.fit_on_texts(df['cleaned_text'])
#Goes through all your cleaned text and builds a word index (a dictionary of word → ID).

sequences=tokenizer.texts_to_sequences(df['cleaned_text'])
#Converts each sentence into a list of word IDs.

padded=pad_sequences(sequences,maxlen=100,padding='post',truncating='post')
#Ensures all sequences are exactly 100 tokens long (Bi-LSTM expects same-length inputs).
#If a sentence is:
#Shorter than 100 → it adds 0s at the end (post-padding)
#Longer than 100 → it cuts off extra words at the end (post-truncating)



#Splitting data for training and testing

array([[  48,  445, 4349, ...,    0,    0,    0],
       [  49,  314, 1405, ...,    0,    0,    0],
       [  51,  462,   10, ...,    0,    0,    0],
       ...,
       [   1,    1, 4310, ...,    0,    0,    0],
       [ 308,  529,  640, ...,    0,    0,    0],
       [  14, 3323,  271, ...,    0,    0,    0]], dtype=int32)

In [12]:

X=padded
y=df['label'].values

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#Building Bi-LSTM model

In [13]:
model=Sequential([
    Embedding(input_dim=5000,output_dim=128),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [68]:
#Embedding(input_dim=10000, output_dim=128)
'''
-Converts each word (integer ID) into a dense vector (word embedding).
-input_dim=10000: only the top 10k most common words are considered.
-output_dim=128: each word will be represented as a 128-dimensional vector.
(Word vectors are often referred to as "embeddings".)'''
#Bidirectional(LSTM(64))
'''
-This is your Bi-LSTM layer, the heart of your model!
-It reads the sentence from both directions (forward and backward).
-LSTM(64) gives you 64 hidden units in each direction, so 128 total features.
(Context understanding + sentiment analysis (both forward and backward).)'''
#Dropout(0.5)
'''
-Randomly drops 50% of neurons during training to prevent overfitting.
-Helps generalize better to unseen data.
(Regularization to prevent overfitting)'''
#Dense(32, activation='relu')
'''
-A regular fully connected layer with 32 neurons.
-ReLU helps learn non-linear patterns.
-Think of this as your feature combiner before output.
(Combining features to learn complex patterns)'''
#Dense(1, activation='sigmoid')
'''
-The output layer.
-Returns a single number between 0 and 1 → representing probability of spam.
-Perfect for binary classification like spam vs ham.
(Binary classification output layer)
'''
'''
Text → Tokenized → Padded → Embedding → Bi-LSTM → Dropout → Dense → Output (0 or 1)
'''

'\nText → Tokenized → Padded → Embedding → Bi-LSTM → Dropout → Dense → Output (0 or 1)\n'

#Compile and Training

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
es=EarlyStopping(patience=5,restore_best_weights=True,monitor='val_loss')

In [16]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
history=model.fit(X_train,y_train,
                  validation_data=(X_test,y_test),
                  epochs=30,
                  batch_size=32,callbacks=[es])


Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.9983 - loss: 0.0085 - val_accuracy: 0.9875 - val_loss: 0.0854
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9996 - loss: 0.0015 - val_accuracy: 0.9786 - val_loss: 0.1050
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9981 - loss: 0.0066 - val_accuracy: 0.9821 - val_loss: 0.1088
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9991 - loss: 0.0026 - val_accuracy: 0.9848 - val_loss: 0.0918
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 1.0000 - loss: 2.4761e-04 - val_accuracy: 0.9857 - val_loss: 0.1014
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 1.0000 - loss: 2.1225e-04 - val_accuracy: 0.9866 - val_loss: 0.1112


In [70]:
#Model.compile->tells the model how to be trained
'''
loss: tells the model how wrong it is — and guides learning by minimizing this loss.
optimizer: Adam is an optimization algorithm that updates weights efficiently
metrics: This tells Keras to track accuracy during training and validation.
'''
#Model.fit->fits the model to the data,starts the training process
'''
-X_train, y_train: The training data used to teach the model.
-validation_data=(X_test, y_test): After each epoch, the model evaluates on this test set to see how it's generalizing.
-epochs=5: The model will go through the entire dataset 5 times.
-batch_size=32: Model updates its weights every 32 samples.This balances speed and stability.
'''
'''---'''


'---'

#Evaluation and Prediction

In [17]:
y_pred=(model.predict(X_test)>0.5).astype(int)
print(classification_report(y_test,y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       954
           1       0.98      0.93      0.96       165

    accuracy                           0.99      1119
   macro avg       0.98      0.97      0.97      1119
weighted avg       0.99      0.99      0.99      1119



#making predictions on new text

In [18]:
def predict_spam(text):
  cleaned=clean_text(text)
  seq=tokenizer.texts_to_sequences([cleaned])
  padded_seq=pad_sequences(seq,maxlen=100,padding='post',truncating='post')
  pred=model.predict(padded_seq)[0][0]
  return "Spam" if pred>0.5 else "Ham"


In [19]:

print(predict_spam("Congratulations! You won a free trip to the beach."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Spam


In [20]:
# Save model in old H5 format (works with TF/Keras 2.x)
model.save("spam_ham_classifier_bilstm.keras")  # .keras format recommended


# Save tokenizer in JSON (cross-version compatible)
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)



In [21]:
from google.colab import files
files.download("spam_ham_classifier_bilstm.keras")
files.download("tokenizer.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>